# День 7 - Анализ ошибок и мини-приложение

Цель проанализировать ошибки fine-tuned DistilBERT и подготовить
модель для демонстрации через Gradio.

Используем ту же validation выборку из 400 объектов,
что и в Днях 5-6.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

In [3]:
MAX_LENGTH = 128
BATCH_SIZE = 32

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

data_dir = Path("../data")
models_dir = Path("../models")
outputs_dir = Path("../outputs")

validation_path = (
    data_dir
    / "splits"
    / "validation.csv"
)

fine_tuned_dir = (
    models_dir
    / "fine_tuned_model"
)

error_analysis_dir = (
    outputs_dir
    / "error_analysis"
)

error_analysis_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print("Device:", device)
print("Validation:", validation_path.resolve())
print("Fine-tuned model:", fine_tuned_dir.resolve())
print("Error analysis output:", error_analysis_dir.resolve())

Device: cpu
Validation: C:\Users\User\Projects\transformers_overall\data\splits\validation.csv
Fine-tuned model: C:\Users\User\Projects\transformers_overall\models\fine_tuned_model
Error analysis output: C:\Users\User\Projects\transformers_overall\outputs\error_analysis


In [4]:
validation_df = pd.read_csv(
    validation_path
)

print("Validation shape:", validation_df.shape)

print("\nClasses:")
print(
    validation_df["label"]
    .value_counts()
    .sort_index()
)

display(validation_df.head())

Validation shape: (400, 2)

Classes:
label
0    200
1    200
Name: count, dtype: int64


,text,label
0,"stale , standard , connect-the-dots storyline",0
1,feel the wasted potential of this slapstick co...,0
2,"is smart to vary the pitch of his movie , bala...",1
3,watch people doing unpleasant things to each o...,0
4,corny television,0


In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    fine_tuned_dir
)

model_ft = (
    AutoModelForSequenceClassification
    .from_pretrained(
        fine_tuned_dir
    )
)

model_ft = model_ft.to(device)
model_ft.eval()

print("Fine-tuned model loaded")
print("Device:", next(model_ft.parameters()).device)
print("Number of labels:", model_ft.config.num_labels)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5906.51it/s]

Fine-tuned model loaded
Device: cpu
Number of labels: 2


In [6]:
def predict_fine_tuned(
    texts,
    model,
    tokenizer,
    device,
    batch_size=32,
):
    if isinstance(texts, str):
        texts = [texts]

    model.eval()

    results = []

    for start_idx in range(
        0,
        len(texts),
        batch_size,
    ):
        batch_texts = texts[
            start_idx:start_idx + batch_size
        ]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=1,
        )

        predictions = torch.argmax(
            probabilities,
            dim=1,
        )

        probabilities = (
            probabilities
            .cpu()
            .numpy()
        )

        predictions = (
            predictions
            .cpu()
            .numpy()
        )

        for text, prediction, probs in zip(
            batch_texts,
            predictions,
            probabilities,
        ):
            results.append({
                "text": text,
                "prediction": int(prediction),
                "negative_probability": float(probs[0]),
                "positive_probability": float(probs[1]),
            })

    return results

In [7]:
validation_texts = (
    validation_df["text"]
    .tolist()
)

predictions = predict_fine_tuned(
    validation_texts,
    model=model_ft,
    tokenizer=tokenizer,
    device=device,
    batch_size=BATCH_SIZE,
)

print(
    "Predictions:",
    len(predictions),
)

Predictions: 400


In [8]:
df_test = pd.DataFrame({
    "text": validation_df["text"],
    "true_label": validation_df["label"],
    "pred_label": [
        result["prediction"]
        for result in predictions
    ],
    "negative_probability": [
        result["negative_probability"]
        for result in predictions
    ],
    "positive_probability": [
        result["positive_probability"]
        for result in predictions
    ],
})

In [9]:
df_test["confidence"] = df_test[
    [
        "negative_probability",
        "positive_probability",
    ]
].max(axis=1)

display(df_test.head())

,text,true_label,pred_label,negative_probability,positive_probability,confidence
0,"stale , standard , connect-the-dots storyline",0,0,0.997081,0.002919,0.997081
1,feel the wasted potential of this slapstick co...,0,0,0.996698,0.003302,0.996698
2,"is smart to vary the pitch of his movie , bala...",1,1,0.003803,0.996197,0.996197
3,watch people doing unpleasant things to each o...,0,0,0.987906,0.012094,0.987906
4,corny television,0,0,0.996712,0.003288,0.996712


In [10]:
errors = df_test[
    df_test["true_label"]
    != df_test["pred_label"]
].copy()

fp = errors[
    (errors["true_label"] == 0)
    & (errors["pred_label"] == 1)
].copy()

fn = errors[
    (errors["true_label"] == 1)
    & (errors["pred_label"] == 0)
].copy()

print("Total errors:", len(errors))
print("False Positives:", len(fp))
print("False Negatives:", len(fn))

Total errors: 44
False Positives: 19
False Negatives: 25


In [11]:
df_test["text_length"] = (
    df_test["text"]
    .str.len()
)

errors["text_length"] = (
    errors["text"]
    .str.len()
)

fp["text_length"] = (
    fp["text"]
    .str.len()
)

fn["text_length"] = (
    fn["text"]
    .str.len()
)

In [12]:
average_error_length = (
    errors["text_length"].mean()
)

average_all_length = (
    df_test["text_length"].mean()
)

print(
    "Average error text length:",
    f"{average_error_length:.1f}",
)

print(
    "Average all text length:",
    f"{average_all_length:.1f}",
)

print(
    "Average FP text length:",
    f"{fp['text_length'].mean():.1f}",
)

print(
    "Average FN text length:",
    f"{fn['text_length'].mean():.1f}",
)

Average error text length: 46.4
Average all text length: 54.1
Average FP text length: 49.4
Average FN text length: 44.1


In [13]:
print("FALSE POSITIVES")
print()

for _, row in fp.head(5).iterrows():
    print("Text:", row["text"])

    print(
        "True label:",
        row["true_label"],
        "negative",
    )

    print(
        "Predicted:",
        row["pred_label"],
        "positive",
    )

    print(
        "Positive probability:",
        f"{row['positive_probability']:.4f}",
    )

    print(
        "Text length:",
        row["text_length"],
    )

    print()

FALSE POSITIVES

Text: could have been much better
True label: 0 negative
Predicted: 1 positive
Positive probability: 0.9895
Text length: 27

Text: sheridan 's take on the author 's schoolboy memoir ... is a rather toothless take on a hard young life .
True label: 0 negative
Predicted: 1 positive
Positive probability: 0.9027
Text length: 104

Text: its digs at modern society are all things we 've seen before .
True label: 0 negative
Predicted: 1 positive
Positive probability: 0.9966
Text length: 62

Text: puzzle his most ardent fans
True label: 0 negative
Predicted: 1 positive
Positive probability: 0.9965
Text length: 27

Text: increasingly diverse
True label: 0 negative
Predicted: 1 positive
Positive probability: 0.9949
Text length: 20



In [14]:
print("FALSE NEGATIVES")
print()

for _, row in fn.head(5).iterrows():
    print("Text:", row["text"])

    print(
        "True label:",
        row["true_label"],
        "positive",
    )

    print(
        "Predicted:",
        row["pred_label"],
        "negative",
    )

    print(
        "Negative probability:",
        f"{row['negative_probability']:.4f}",
    )

    print(
        "Text length:",
        row["text_length"],
    )

    print()

FALSE NEGATIVES

Text: ends or just ca n't tear himself away from the characters
True label: 1 positive
Predicted: 0 negative
Negative probability: 0.8930
Text length: 57

Text: may be more genial than ingenious , but it gets the job done
True label: 1 positive
Predicted: 0 negative
Negative probability: 0.7937
Text length: 60

Text: even the bull gets recycled .
True label: 1 positive
Predicted: 0 negative
Negative probability: 0.9963
Text length: 29

Text: never mind all that ; the boobs are fantasti
True label: 1 positive
Predicted: 0 negative
Negative probability: 0.9845
Text length: 44

Text: the screenplay or something
True label: 1 positive
Predicted: 0 negative
Negative probability: 0.9748
Text length: 27



In [15]:
confident_errors = (
    errors
    .sort_values(
        "confidence",
        ascending=False,
    )
)

display(
    confident_errors[
        [
            "text",
            "true_label",
            "pred_label",
            "negative_probability",
            "positive_probability",
            "confidence",
            "text_length",
        ]
    ].head(10)
)

,text,true_label,pred_label,negative_probability,positive_probability,confidence,text_length
165,puzzle his most ardent fans .,0,1,0.003226,0.996774,0.996774,29
238,had all its vital essence scooped,0,1,0.003275,0.996725,0.996725,33
69,its digs at modern society are all things we '...,0,1,0.003400,0.996600,0.996600,62
263,general family chaos to which anyone can relate,1,0,0.996586,0.003414,0.996586,47
86,puzzle his most ardent fans,0,1,0.003475,0.996525,0.996525,27
36,even the bull gets recycled .,1,0,0.996326,0.003674,0.996326,29
336,"crack you up with her crass , then gasp for ga...",1,0,0.996163,0.003837,0.996163,67
288,is the kathie lee gifford of film directors,0,1,0.004096,0.995904,0.995904,43
198,peculiarly,0,1,0.004138,0.995862,0.995862,10
183,it all unfolds predictably,0,1,0.004155,0.995845,0.995845,26


In [16]:
print("Error analysis summary")
print("Total validation samples:", len(df_test))
print("Total errors:", len(errors))
print("False Positives:", len(fp))
print("False Negatives:", len(fn))

print()

print(
    "Average all text length:",
    f"{average_all_length:.1f}",
)

print(
    "Average error text length:",
    f"{average_error_length:.1f}",
)

print(
    "Average FP text length:",
    f"{fp['text_length'].mean():.1f}",
)

print(
    "Average FN text length:",
    f"{fn['text_length'].mean():.1f}",
)

Error analysis summary
Total validation samples: 400
Total errors: 44
False Positives: 19
False Negatives: 25

Average all text length: 54.1
Average error text length: 46.4
Average FP text length: 49.4
Average FN text length: 44.1


In [17]:
error_analysis_path = (
    error_analysis_dir
    / "error_analysis.txt"
)

In [19]:
error_analysis_text = f"""Transformers Day 7 - Error Analysis
===================================

Summary
-------
Validation samples: {len(df_test)}
Total errors: {len(errors)}
False Positives: {len(fp)}
False Negatives: {len(fn)}

Text length analysis
--------------------
Average all text length: {average_all_length:.1f}
Average error text length: {average_error_length:.1f}
Average FP text length: {fp["text_length"].mean():.1f}
Average FN text length: {fn["text_length"].mean():.1f}

False Positive examples
-----------------------
"""

for _, row in fp.head(5).iterrows():
    error_analysis_text += (
        f"\nText: {row['text']}\n"
        f"True label: negative\n"
        f"Predicted label: positive\n"
        f"Positive probability: "
        f"{row['positive_probability']:.4f}\n"
        f"Text length: {row['text_length']}\n"
    )

error_analysis_text += """

False Negative examples
-----------------------
"""

for _, row in fn.head(5).iterrows():
    error_analysis_text += (
        f"\nText: {row['text']}\n"
        f"True label: positive\n"
        f"Predicted label: negative\n"
        f"Negative probability: "
        f"{row['negative_probability']:.4f}\n"
        f"Text length: {row['text_length']}\n"
    )

error_analysis_text += f"""

Observations
------------
1. Error texts are shorter on average than the full validation set.
   Average error length is {average_error_length:.1f} characters,
   compared with {average_all_length:.1f} characters for all texts.

2. False Negative texts are the shortest error group on average
   ({fn["text_length"].mean():.1f} characters). Some errors occur on
   short or contextually incomplete phrases where sentiment is difficult
   to determine from the fragment alone.

3. Indirect, mixed and contrastive sentiment can cause errors.
   Phrases containing constructions such as "but" or conditional wording
   may contain both positive and negative signals.

4. Individual sentiment-bearing words can be misleading when their meaning
   depends on the surrounding context. For example, a positive-looking word
   such as "better" does not necessarily make the whole phrase positive.

5. Some incorrect predictions have very high confidence. This shows that
   a high softmax probability does not guarantee that the predicted class
   is correct.

6. The model produced more False Negatives than False Positives:
   {len(fn)} versus {len(fp)}. It therefore made slightly more errors where
   positive texts were classified as negative.
"""

In [20]:
error_analysis_path.write_text(
    error_analysis_text,
    encoding="utf-8",
)

print(error_analysis_text)

print(
    "Error analysis saved:",
    error_analysis_path.resolve(),
)

Transformers Day 7 - Error Analysis

Summary
-------
Validation samples: 400
Total errors: 44
False Positives: 19
False Negatives: 25

Text length analysis
--------------------
Average all text length: 54.1
Average error text length: 46.4
Average FP text length: 49.4
Average FN text length: 44.1

False Positive examples
-----------------------

Text: could have been much better
True label: negative
Predicted label: positive
Positive probability: 0.9895
Text length: 27

Text: sheridan 's take on the author 's schoolboy memoir ... is a rather toothless take on a hard young life .
True label: negative
Predicted label: positive
Positive probability: 0.9027
Text length: 104

Text: its digs at modern society are all things we 've seen before .
True label: negative
Predicted label: positive
Positive probability: 0.9966
Text length: 62

Text: puzzle his most ardent fans
True label: negative
Predicted label: positive
Positive probability: 0.9965
Text length: 27

Text: increasingly diverse
True 

In [21]:
errors_path = (
    error_analysis_dir
    / "error_examples.csv"
)

errors.to_csv(
    errors_path,
    index=False,
    encoding="utf-8",
)

print(
    "Error examples saved:",
    errors_path.resolve(),
)

Error examples saved: C:\Users\User\Projects\transformers_overall\outputs\error_analysis\error_examples.csv
